# Week 8 - 비지도학습 실습(고급): 서울시 따릉이 대여소 공간 군집화 (실제 데이터)

## 🚲 서울시설공단 따릉이 운영팀의 고민

당신은 **서울시설공단 따릉이 운영팀의 데이터 사이언티스트**다. 서울 전역에 3,000개 이상의 따릉이 대여소가 운영 중이며, 매일 **자전거 재배치 트럭**이 수요-공급 불균형을 해소한다. 대여소별 위치와 월별 대여 건수 데이터를 활용해 두 가지 **서로 다른** 운영 의사결정을 지원해야 한다.

| # | 해결하고자 하는 문제/질문 | 적합 알고리즘 | 이유 |
|---|---|---|---|
| **Q1** | 재배치 트럭 **관리 구역을 몇 개로 나눠** 각 구역당 한 대씩 배치할까? 각 트럭의 주 거점은 어디에? | **K-Means** | 공간을 균등하게 분할, centroid = 트럭 차고지 후보 |
| **Q2** | 시민들이 자연스럽게 만들어내는 **수요 핫스팟(상권·교통허브)** 은 어디이고, 이용량이 적어 운영 효율이 낮은 **저이용 대여소**는 어디인가? | **DBSCAN** | 밀도 기반 자연 클러스터 + 노이즈(저이용 대여소) 식별 |

## 📊 데이터셋
- **출처**: 서울 열린데이터광장 (data.seoul.go.kr) 에서 다운로드한 2개 파일을 merge 하여 본 실습을 위한 하나의 csv 파일로 가공함
  - [따릉이 대여소 마스터 정보](https://data.seoul.go.kr/dataList/OA-21235/S/1/datasetView.do) (위도/경도)
  - [대여소별 월별 이용정보](https://data.seoul.go.kr/dataList/OA-15249/F/1/datasetView.do) (월 대여/반납 건수)
- **기간**: 2025년 7~11월 (5개월)
- **규모**: 1,760개 대여소 (25개 자치구 전역)
- **주요 컬럼**: `station_id`, `lat`, `lon`, `monthly_usage`, `gu`, `rent_202507`~`rent_202511`

## 🎯 학습 목표
1. **대여소 단위 데이터**(point data with attribute)에 군집화를 적용
2. **Haversine 거리**(실제 지구/지표면상에서의 포인트간 거리 계산에 사용)로 지리 데이터에 DBSCAN을 올바르게 적용
3. **이용량을 가중치(sample_weight)** 로 사용하는 `Weighted K-Means / Weighted DBSCAN` 실습
4. 실제 서울 데이터에서 발생하는 **데이터 품질 이슈**(결측, 이상값, 마스터-이용정보 미스매치)를 직접 경험
5. 하이퍼파라미터 선택 및 평가

---

---
## 1. 환경 설정 및 데이터 로딩


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# heatmap 시각화를 위한 라이브러리 설치/임포트
!pip install folium -q

import folium
from folium.plugins import HeatMap

Colab 환경에서 한글 폰트 설정을 위한 라이브러리 설치
(한글이 깨져서 나타날 경우 설치)

In [ ]:
!pip install koreanize-matplotlib

import koreanize_matplotlib

In [ ]:
# ==== 데이터 로딩 ====

url = "https://raw.githubusercontent.com/agtechresearch/MLapplications-graduate/refs/heads/main/dataset/SeoulBike_dataset.csv"
df = pd.read_csv(url)

In [ ]:
print(f'✅ 로드 완료: {len(df):,}개 대여소')
print(f'컬럼: {list(df.columns)}')
df.head()

## 1. 탐색적 데이터 분석 (EDA)

실제 데이터이므로 먼저 데이터 특성을 이해한다.

In [ ]:
print(f'총 대여소 수: {len(df):,}')
print(f'자치구 수: {df["gu"].nunique()}')
print(f'위도 범위: [{df.lat.min():.4f}, {df.lat.max():.4f}]')
print(f'경도 범위: [{df.lon.min():.4f}, {df.lon.max():.4f}]')

print(f'\n월 평균 대여량 통계:')
print(df['monthly_usage'].describe().round(0))

print(f'\n🏆 월 평균 대여량 TOP 10 대여소:')
print(df[['station_name', 'gu', 'monthly_usage']].head(10).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sc = axes[0].scatter(df['lon'], df['lat'],
                     s=df['monthly_usage']/100,
                     c=df['monthly_usage'],
                     cmap='YlOrRd', alpha=0.6,
                     edgecolor='black', linewidth=0.15)
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
axes[0].set_title(f'대여소 분포 (n={len(df):,})\n크기/색 = 월 평균 대여량')
plt.colorbar(sc, ax=axes[0], label='Monthly Usage')

axes[1].hist(df['monthly_usage'], bins=60, color='#2E86AB', edgecolor='black')
axes[1].set_xlabel('Monthly Usage'); axes[1].set_ylabel('# of Stations')
axes[1].set_title('이용량 분포 (Long-tail)')
axes[1].set_yscale('log')
axes[1].axvline(df['monthly_usage'].median(), color='red', linestyle='--',
                label=f'median={df["monthly_usage"].median():.0f}')
axes[1].legend()

gu_stats = df.groupby('gu').agg(n=('station_id', 'count')).sort_values('n', ascending=True)
axes[2].barh(gu_stats.index, gu_stats['n'], color='#A23B72', alpha=0.7)
axes[2].set_xlabel('대여소 수'); axes[2].set_title('자치구별 대여소 수')
axes[2].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()

**관찰 포인트**
- **Long-tail 분포**: 대부분 월 1,000건 수준이지만 상위 몇 개는 10,000건 이상 (예: 마곡나루역, 잠실 롯데월드타워)
- **자치구별 편차**: 송파·서초·강남·강서·영등포가 상위권
- 이 구조가 **단순 공간 군집화와 가중치 군집화가 크게 다른 결과**를 만들어낸다

## 2. [Part 1] K-Means로 재배치 트럭 관리 구역 설계 (Q1)

> **비즈니스 질문**: 재배치 트럭 $k$대를 어디에 배치할까?

**두 가지 접근을 비교한다:**
- **(a) Vanilla K-Means**: 모든 대여소를 동등하게 → 공간적으로 균등한 구역
- **(b) Weighted K-Means** (`sample_weight=monthly_usage`): 이용량 많은 대여소에 가중치 → centroid가 수요 많은 곳으로 끌려감

In [ ]:
coords = df[['lat', 'lon']].values
weights = df['monthly_usage'].values

ks = range(2, 16)
inertias = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(coords)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(ks), inertias, 'o-', linewidth=2, markersize=8)
plt.axvline(x=8, color='red', linestyle='--', alpha=0.5, label='후보 k=8')
plt.xlabel('k (트럭 대수)'); plt.ylabel('Inertia')
plt.title('Elbow Method — 몇 대의 재배치 트럭이 필요한가?')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print('💡 실제 데이터에서는 elbow가 뚜렷하지 않음 — 서울이 전반적으로 연속 분포이기 때문.')
print('   운영 관점에서 "트럭 1대가 담당 가능한 규모"를 함께 고려해 결정해야 함.')

In [ ]:
K = 8

km_v = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(coords)
labels_km_v = km_v.labels_
centers_v = km_v.cluster_centers_

km_w = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(coords, sample_weight=weights)
labels_km_w = km_w.labels_
centers_w = km_w.cluster_centers_

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
palette = plt.cm.tab10(np.linspace(0, 1, K))

for ax, labels, centers, title in [
    (axes[0], labels_km_v, centers_v, '(a) Vanilla K-Means\n모든 대여소를 동등하게'),
    (axes[1], labels_km_w, centers_w, '(b) Weighted K-Means (sample_weight=monthly_usage)\n이용량 많은 곳으로 끌림'),
]:
    for k in range(K):
        m = labels == k
        ax.scatter(df.loc[m, 'lon'], df.loc[m, 'lat'],
                   s=df.loc[m, 'monthly_usage']/120, alpha=0.5, color=palette[k],
                   edgecolor='black', linewidth=0.1)
    ax.scatter(centers[:, 1], centers[:, 0],
               c='red', marker='X', s=400, edgecolor='white', linewidth=2, zorder=5)
    for i, (lat, lon) in enumerate(centers):
        ax.annotate(f'T{i}', (lon, lat), fontsize=11, fontweight='bold',
                    xytext=(8, 8), textcoords='offset points')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_title(title)

plt.tight_layout()
plt.show()

print('\n각 트럭 거점 이동 거리 (Vanilla → Weighted):')
for i in range(K):
    dists = np.linalg.norm(centers_v - centers_w[i], axis=1) * 111
    j = np.argmin(dists)
    print(f'  Truck {i}:  {dists[j]:.2f} km 이동')

### 💡 K-Means 결과 해석
- **Vanilla**: 공간 중심(geometric center)에 centroid → 외곽 지역까지 균등 포함
- **Weighted**: centroid가 강남·마곡·잠실 등으로 끌려감
- **운영적 시사점**:
  - 차고지 목적이 "면적 커버"라면 vanilla
  - "대여량 대응 속도"라면 weighted가 적합
- 실무 재배치 문제는 대부분 **weighted가 더 적절**

In [ ]:
df['zone_weighted'] = labels_km_w
zone_analysis = df.groupby('zone_weighted').agg(
    n_stations=('station_id', 'count'),
    total_usage=('monthly_usage', 'sum'),
    top_gu=('gu', lambda x: x.mode()[0]),
    hub_lat=('lat', 'mean'),
    hub_lon=('lon', 'mean'),
).round(2).sort_values('total_usage', ascending=False)

print('🚛 Weighted K-Means 각 트럭 구역 분석:')
print(zone_analysis.to_string())

## 3. [Part 2] DBSCAN으로 수요 핫스팟 탐지 (Q2)

> **비즈니스 질문**: 시민들이 자연스럽게 형성한 수요 군집은 어디인가? 저이용 대여소는?

### 3-1. Haversine 거리 설정

지리 데이터는 구면 위에 있으므로 **haversine 거리**(실제 지표면 거리)를 사용한다.

In [ ]:
EARTH_RADIUS_KM = 6371.0088
coords_rad = np.radians(coords)

def km_to_rad(km):
    return km / EARTH_RADIUS_KM

### 3-2. k-distance graph로 eps 선택

In [ ]:
MIN_SAMPLES = 10

nn = NearestNeighbors(n_neighbors=MIN_SAMPLES, metric='haversine').fit(coords_rad)
dists, _ = nn.kneighbors(coords_rad)
k_dists_m = np.sort(dists[:, -1]) * EARTH_RADIUS_KM * 1000

plt.figure(figsize=(8, 4))
plt.plot(k_dists_m, linewidth=2)
plt.axhline(y=800, color='red', linestyle='--', alpha=0.6, label='eps = 800m')
plt.xlabel('points (sorted)')
plt.ylabel(f'{MIN_SAMPLES}-th nearest neighbor distance (m)')
plt.title(f'k-distance graph (min_samples={MIN_SAMPLES})')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print(f'\n💡 서울은 대여소 밀도가 균질해서 elbow가 부드러움.')
print(f'   중앙값 {np.median(k_dists_m):.0f}m, 80% 지점 {np.percentile(k_dists_m, 80):.0f}m')
print(f'   → 약 800m를 eps로 선택')

### 3-3. Vanilla vs Weighted DBSCAN

In [ ]:
EPS_M = 800
eps_rad = km_to_rad(EPS_M / 1000)

db_v = DBSCAN(eps=eps_rad, min_samples=MIN_SAMPLES, metric='haversine').fit(coords_rad)
labels_db_v = db_v.labels_

db_w = DBSCAN(eps=eps_rad, min_samples=MIN_SAMPLES, metric='haversine')
db_w.fit(coords_rad, sample_weight=weights / weights.mean())
labels_db_w = db_w.labels_

def summarize(labels, name):
    n_c = len(set(labels)) - (1 if -1 in labels else 0)
    n_n = (labels == -1).sum()
    print(f'{name:30s}: {n_c}개 핫스팟, 노이즈 {n_n}개 ({n_n/len(labels)*100:.1f}%)')

summarize(labels_db_v, '(a) Vanilla DBSCAN')
summarize(labels_db_w, '(b) Weighted DBSCAN')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, labels, title in [
    (axes[0], labels_db_v, f'(a) Vanilla DBSCAN (eps={EPS_M}m)'),
    (axes[1], labels_db_w, f'(b) Weighted DBSCAN (sample_weight=이용량)'),
]:
    mn = labels == -1
    ax.scatter(df.loc[mn, 'lon'], df.loc[mn, 'lat'],
               s=15, alpha=0.4, color='lightgray', marker='x',
               label=f'저이용/외곽 ({mn.sum()})')
    uc = sorted(set(labels) - {-1})
    colors = plt.cm.turbo(np.linspace(0.05, 0.95, max(len(uc), 1)))
    for c, color in zip(uc, colors):
        m = labels == c
        ax.scatter(df.loc[m, 'lon'], df.loc[m, 'lat'],
                   s=df.loc[m, 'monthly_usage']/100, alpha=0.7, color=color,
                   edgecolor='black', linewidth=0.1)
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_title(title)
    ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

### 💡 DBSCAN 결과 해석
- **Vanilla**: 공간 밀도만 고려 → 자치구 경계에 가까운 자연스러운 핫스팟
- **Weighted**: 이용량 높은 대여소가 core point가 되기 쉬움 → 더 많은 핫스팟 발견
- 두 결과 모두 **외곽·경사 지역**(관악, 강북)에서 노이즈가 집중

## 4. 핫스팟별 운영 분석

In [ ]:
df['hotspot'] = labels_db_w

hotspot_summary = df[df['hotspot'] != -1].groupby('hotspot').agg(
    n_stations=('station_id', 'count'),
    total_usage=('monthly_usage', 'sum'),
    avg_usage=('monthly_usage', 'mean'),
    top_gu=('gu', lambda x: x.mode()[0]),
    center_lat=('lat', 'mean'),
    center_lon=('lon', 'mean'),
).round(1).sort_values('total_usage', ascending=False)

print('🔥 핫스팟별 운영 통계 (Weighted DBSCAN, TOP 15):')
print(hotspot_summary.head(15).to_string())

print(f"\n📉 노이즈(저이용 외곽) 대여소: {(df['hotspot'] == -1).sum()}개")
print(f"   평균 월 이용량: {df.loc[df['hotspot'] == -1, 'monthly_usage'].mean():.0f}건")
print(f"   (참고) 전체 평균: {df['monthly_usage'].mean():.0f}건")

print('\n🌄 노이즈가 많은 자치구 TOP 5 (저이용 대여소 밀집):')
noise_by_gu = df.groupby('gu').apply(
    lambda g: (g['hotspot'] == -1).sum() / len(g) * 100, include_groups=False
).sort_values(ascending=False).head(5)
for gu, pct in noise_by_gu.items():
    print(f'   {gu}: {pct:.1f}% 노이즈 비율')

> **관찰**: 관악·강북·중구·용산이 노이즈 비율 상위 — 산지·성곽지구 등 **지형적 요인**과 **연속 대여소 네트워크의 단절**이 주 원인. 비즈니스 관점에서는 "폐쇄 대상"이 아니라 "교통 소외 지역 서비스"로 해석해야 한다.

## 5. 두 알고리즘 직접 비교

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

ax = axes[0]
for k in range(K):
    m = labels_km_w == k
    ax.scatter(df.loc[m, 'lon'], df.loc[m, 'lat'],
               s=df.loc[m, 'monthly_usage']/120, alpha=0.5, color=palette[k])
ax.scatter(centers_w[:, 1], centers_w[:, 0],
           c='red', marker='X', s=400, edgecolor='white', linewidth=2, zorder=5)
ax.set_title(f'Weighted K-Means (k={K})\n→ Q1: 재배치 트럭 {K}대의 차고지 + 관리 구역')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

ax = axes[1]
mn = labels_db_w == -1
ax.scatter(df.loc[mn, 'lon'], df.loc[mn, 'lat'],
           s=15, alpha=0.4, color='lightgray', marker='x')
uc = sorted(set(labels_db_w) - {-1})
colors = plt.cm.turbo(np.linspace(0.05, 0.95, max(len(uc), 1)))
for c, color in zip(uc, colors):
    m = labels_db_w == c
    ax.scatter(df.loc[m, 'lon'], df.loc[m, 'lat'],
               s=df.loc[m, 'monthly_usage']/100, alpha=0.7, color=color)
ax.set_title(f'Weighted DBSCAN (eps={EPS_M}m)\n→ Q2: 자연 수요 핫스팟 {len(uc)}개 + 저이용 대여소 식별')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

plt.tight_layout()
plt.show()

### 📋 두 결과의 비즈니스적 의미

| 기준 | Weighted K-Means | Weighted DBSCAN |
|---|---|---|
| 비즈니스 용도 | 재배치 트럭 배치 & 구역 책임 | 핫스팟 마케팅 / 저이용 대여소 발굴 |
| 모든 대여소 할당 | 100% (강제) | 핫스팟 or 노이즈 |
| 클러스터 개수 | 경영 결정 (k=트럭 대수) | 데이터가 결정 |
| 저이용 대여소 | 어느 한 트럭에 할당 | 노이즈로 자동 식별 |

**핵심 메시지**: **서로 다른 경영 질문에 서로 다른 답.** 실무에서는 둘 다 동시에 활용한다.

## 6. Folium 인터랙티브 지도

In [ ]:
m = folium.Map(location=[37.55, 126.98], zoom_start=11, tiles='CartoDB positron')

heat_layer = folium.FeatureGroup(name='이용량 Heatmap').add_to(m)
HeatMap(df[['lat', 'lon', 'monthly_usage']].values.tolist(),
        radius=10, blur=14, min_opacity=0.3).add_to(heat_layer)

truck_layer = folium.FeatureGroup(name='K-Means 트럭 차고지').add_to(m)
for i, (lat, lon) in enumerate(centers_w):
    folium.Marker(
        location=[lat, lon],
        popup=f'<b>Truck Base {i}</b>',
        icon=folium.Icon(color='red', icon='truck', prefix='fa'),
    ).add_to(truck_layer)

hotspot_layer = folium.FeatureGroup(name='DBSCAN 핫스팟').add_to(m)
for _, row in hotspot_summary.iterrows():
    folium.CircleMarker(
        location=[row['center_lat'], row['center_lon']],
        radius=np.sqrt(row['total_usage']) / 80,
        popup=f"<b>Hotspot ({row['top_gu']})</b><br>"
              f"{int(row['n_stations'])} 대여소<br>"
              f"월 {int(row['total_usage']):,} 건",
        color='blue', fill=True, fillColor='blue', fillOpacity=0.35,
    ).add_to(hotspot_layer)

noise_layer = folium.FeatureGroup(name='저이용 대여소 (노이즈)', show=False).add_to(m)
for _, row in df[df['hotspot'] == -1].iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=2.5,
        popup=f"<b>{row['station_name']}</b><br>"
              f"{row['gu']}<br>월 {int(row['monthly_usage'])}건",
        color='gray', fill=True, fillColor='gray', fillOpacity=0.5,
    ).add_to(noise_layer)

folium.LayerControl(collapsed=False).add_to(m)
m

- 🔴 **빨간 트럭 아이콘**: Weighted K-Means 재배치 트럭 차고지
- 🔵 **파란 원**: Weighted DBSCAN 핫스팟 (원 크기 = 총 월간 이용량)
- ⚪ **회색 점**: 노이즈(저이용) 대여소 (기본 숨김 — 좌상단에서 켜기)
- 🔥 **배경 Heatmap**: 전체 이용량 밀도

## 7. 과제 및 토론

### 🎯 실습 과제

1. **[월별 변동 분석]** `rent_202507` ~ `rent_202511` 컬럼을 활용해, 각 월별로 Weighted DBSCAN을 따로 돌리고 **핫스팟 구조가 월에 따라 변하는지** 확인하라. 어떤 자치구의 핫스팟이 확장/축소되었는가? 계절성의 영향은?

2. **[하이퍼파라미터 민감도]** `eps`를 500m, 800m, 1200m로, `min_samples`를 5, 10, 20으로 조합(9 케이스)하여 DBSCAN 결과를 비교하라. 각 조합의 **#핫스팟, 노이즈 비율, Top1 크기**를 표로 정리하라.

3. **[자치구 기반 평가]** Weighted K-Means 구역과 실제 **자치구 경계**의 일치도를 정량 평가하라. ARI 또는 NMI를 사용하라.

4. **[HDBSCAN 도전]** `pip install hdbscan` 후 동일 데이터에 HDBSCAN을 적용. 서울 도심(고밀도)과 외곽(저밀도)이 공존하는 상황에서 단일 `eps` 방식보다 어떤 장점이 있는가? `min_cluster_size=20`으로 시작하라.

5. **[수송 문제 최적화]** 각 핫스팟을 수요 노드, 각 트럭 거점을 공급 노드로 보고, `scipy.optimize.linprog`로 최적 재배치 경로를 계산하라. 수요-공급 불균형을 최소화하는 일일 수송량은?

### 💬 토론 질문

1. **Weighted K-Means에서 "가중치가 크다"는 건 그 대여소가 더 자주 서비스 받아야 한다는 뜻인가, 아니면 그 대여소에 더 많은 자전거를 배치해야 한다는 뜻인가?** 재배치 운영의 관점에서 어느 해석이 맞는가?

2. **관악구·강북구에 노이즈 비율이 높다.** 이를 근거로 해당 지역 대여소를 줄이는 정책을 편다면, 어떤 윤리적/형평성 이슈가 발생하는가? 공공서비스 제공자로서의 책임은?

3. **서울의 마곡지구(강서구)는 2020년대 이후 급성장으로 따릉이 이용량이 폭증했다.** 이 실습에서도 마곡나루역이 최상위 핫스팟으로 잡힌다. **과거 데이터(2022년 이전)로 동일 분석**을 했다면 결과가 어떻게 달랐을까? 이는 군집화 기반 인프라 투자 결정의 **시간적 타당성** 문제를 제기한다.

4. **본 실습의 evaluation은 전적으로 시각적/정성적이었다.** 정답 레이블이 없는 상황에서 "좋은 군집화"를 정의하는 방법들(Silhouette, Davies-Bouldin, Gap statistic 등)의 **장단점과 도메인 의존성**을 논하라.

5. **이 문제를 실시간(streaming)으로 확장**한다면? 매 5분마다 실시간 대여 현황이 들어올 때, 매번 전체 DBSCAN을 재실행하는 것은 비효율적이다. DenStream, CluStream 같은 streaming clustering 알고리즘을 조사하라.

---

## 부록 A. Raw 파일에서 merged.csv 재생성하기

서울 열린데이터광장에서 `따릉이 대여소 마스터 정보`와 `대여소별 월별 이용정보` 파일을 직접 받아 조인하려면:

```python
import pandas as pd

master = pd.read_csv('마스터파일.csv', encoding='cp949')
usage  = pd.read_csv('이용정보파일.csv', encoding='cp949')

master['station_num'] = master['대여소_ID'].str.replace('ST-', '', regex=False)
usage['station_num'] = usage['대여소명'].str.extract(r'^(\d+)\.')[0]

master = master[(master['위도'].between(37.40, 37.70)) & 
                (master['경도'].between(126.80, 127.20))]

usage_pivot = usage.pivot_table(index='station_num', columns='기준년월', 
                                 values='대여건수', aggfunc='sum').fillna(0)
usage_pivot.columns = [f'rent_{c}' for c in usage_pivot.columns]

usage_agg = usage.groupby('station_num').agg(
    monthly_usage=('대여건수', 'mean'),
    total_rent=('대여건수', 'sum'),
    total_return=('반납건수', 'sum'),
    n_months=('기준년월', 'nunique'),
    자치구=('자치구', 'first'),
    대여소명=('대여소명', 'first'),
).reset_index().merge(usage_pivot, on='station_num', how='left')

merged = master.merge(usage_agg, on='station_num', how='inner')
merged = merged.rename(columns={
    '위도': 'lat', '경도': 'lon', '자치구': 'gu', '대여소명': 'station_name',
    '대여소_ID': 'station_id',
})
merged.to_csv('ttareungyi_merged.csv', index=False, encoding='utf-8-sig')
```

## 📚 References
- Ester, M., Kriegel, H.P., Sander, J., Xu, X. (1996). *A density-based algorithm for discovering clusters in large spatial databases with noise.* KDD.
- 서울 열린데이터광장: https://data.seoul.go.kr
- Campello, R.J.G.B., et al. (2013). *Density-Based Clustering Based on Hierarchical Density Estimates.* PAKDD. (HDBSCAN)
- sklearn 가중치 클러스터링: `KMeans.fit(sample_weight=...)`, `DBSCAN.fit(sample_weight=...)`